# Combined Amber RAG (ChromaDB archive + PDF FAISS) — POC

Single query → hits **two** vector stores **in parallel** (ThreadPoolExecutor), merges results by similarity score, and feeds the top hits to the LLM. A per-store quota guarantees the PDF manual isn't crowded out by chatty archive emails.

- **Store A** — persistent **Chroma** at `ARCHIVE_DB_PATH`.
- **Store B** — in-memory **FAISS** built once from `PDF_PATH`, cached to `FAISS_INDEX_DIR` so it loads instantly on subsequent runs.

Both stores use `sentence-transformers/all-MiniLM-L6-v2` and L2 (squared) distance, so scores are directly comparable across stores.

## 1) Setup

In [12]:
import json
import os
from concurrent.futures import ThreadPoolExecutor
from typing import List, Tuple

import chromadb
from langchain_community.vectorstores import Chroma, FAISS
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_ollama import ChatOllama

## 2) Configuration
All knobs in one place. Anyone reading the notebook can see what's tunable without scrolling through the pipeline.

In [13]:
# --- Paths ---
ARCHIVE_DB_PATH = "/opt/chromadb/data/prompt_db"
ARCHIVE_COLLECTION = "amber_messages"
PDF_PATH = "Amber25.pdf"
FAISS_INDEX_DIR = "./faiss_pdf_index"

# --- Embedding model (must match what was used to build the stores) ---
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

# --- PDF chunking (only used when (re)building the FAISS index) ---
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 100

# --- Retrieval ---
K_TOTAL = 5      # final number of docs passed to the LLM
K_ARCHIVE = 10   # candidates pulled from the Chroma archive
K_PDF = 10       # candidates pulled from the FAISS PDF index
MIN_PDF = 1      # guarantee at least this many PDF hits in the final top-K

# --- LLM ---
OLLAMA_MODEL = "llama3.1:8b"
TEMPERATURE = 0

# --- Eval output ---
RESULTS_PATH = "rag_results.json"  # query_rag appends each Q/A pair here

## 3) Shared embedding model
Both stores must use the same embedding model for the scores to be comparable.

In [14]:
embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)

## 4) Open both stores

- **Archive store** — persistent Chroma at `/opt/chromadb/data/prompt_db`.
- **PDF store** — in-memory FAISS. On first run, it parses `Amber25.pdf`, embeds the chunks, and saves a FAISS index to `./faiss_pdf_index/`. On every subsequent run, it loads that cached index in < 1s instead of re-embedding ~3,800 chunks.

Delete `./faiss_pdf_index/` to force a rebuild (e.g. when the PDF or chunking settings change).

In [15]:
# --- Store A: archive / emails (persistent Chroma) ---
archive_client = chromadb.PersistentClient(path=ARCHIVE_DB_PATH)
vectorstore_archive = Chroma(
    client=archive_client,
    collection_name=ARCHIVE_COLLECTION,
    embedding_function=embeddings,
)

# --- Store B: PDF (in-memory FAISS with cached index on disk) ---
def _clean_pdf_text(text: str) -> str:
    """Minimal cleanup: collapse whitespace and fix common ligatures."""
    text = " ".join(text.split())
    text = text.replace("\ufb01", "fi").replace("\ufb02", "fl")
    return text


def _build_pdf_chunks(pdf_path: str) -> List[Document]:
    """Load the PDF, clean each page, and split into overlapping chunks."""
    pages = PyPDFLoader(pdf_path).load()
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP, separators=[" "]
    )
    chunks: List[Document] = []
    for page_num, page in enumerate(pages):
        cleaned = _clean_pdf_text(page.page_content)
        if len(cleaned.strip()) < 50:
            continue
        chunks.extend(splitter.create_documents(
            texts=[cleaned],
            metadatas=[{
                **page.metadata,
                "page": page_num + 1,
                "total_pages": len(pages),
                "chunk_method": "recursive_char_splitter",
                "char_count": len(cleaned),
            }],
        ))
    return chunks


if os.path.isdir(FAISS_INDEX_DIR):
    vectorstore_pdf = FAISS.load_local(
        FAISS_INDEX_DIR, embeddings, allow_dangerous_deserialization=True
    )
    print(f"Loaded cached FAISS index from {FAISS_INDEX_DIR}")
else:
    print(f"No cached index at {FAISS_INDEX_DIR} — building from {PDF_PATH} ...")
    pdf_chunks = _build_pdf_chunks(PDF_PATH)
    print(f"  {len(pdf_chunks)} chunks, embedding ...")
    vectorstore_pdf = FAISS.from_documents(pdf_chunks, embeddings)
    vectorstore_pdf.save_local(FAISS_INDEX_DIR)
    print(f"  Saved FAISS index to {FAISS_INDEX_DIR}")

print(f"Archive store: {vectorstore_archive._collection.count()} vectors @ {ARCHIVE_DB_PATH}")
print(f"PDF store:     {vectorstore_pdf.index.ntotal} vectors (FAISS in-memory)")

Loaded cached FAISS index from ./faiss_pdf_index
Archive store: 42935 vectors @ /opt/chromadb/data/prompt_db
PDF store:     3793 vectors (FAISS in-memory)


## 5) Merged retriever — parallel query, score-based merge, PDF quota
Both stores are queried in parallel. Hits are tagged with their store and raw L2 score, then merged: we pin the top `MIN_PDF` PDF hits first (so the manual is always represented), and fill the remaining slots by raw score across both stores. Lower L2 distance = closer match.

In [16]:
def _search(store, query, k):
    return store.similarity_search_with_score(query, k=k)


def hybrid_retrieve(
    query: str,
    k_total: int = K_TOTAL,
    k_archive: int = K_ARCHIVE,
    k_pdf: int = K_PDF,
    min_pdf: int = MIN_PDF,
) -> List[Document]:
    """Query both stores in parallel, merge by L2 distance (lower = closer),
    and return up to k_total docs while guaranteeing at least min_pdf PDF hits.
    """
    # Fire both queries in parallel — they share no state.
    with ThreadPoolExecutor(max_workers=2) as ex:
        fut_archive = ex.submit(_search, vectorstore_archive, query, k_archive)
        fut_pdf     = ex.submit(_search, vectorstore_pdf,     query, k_pdf)
        archive_hits = fut_archive.result()
        pdf_hits     = fut_pdf.result()

    # Tag every doc with its store and raw score so downstream code can see it.
    archive_scored: List[Tuple[Document, float]] = []
    for doc, score in archive_hits:
        doc.metadata = {**doc.metadata, "store": "archive", "score": float(score)}
        archive_scored.append((doc, score))

    pdf_scored: List[Tuple[Document, float]] = []
    for doc, score in pdf_hits:
        doc.metadata = {**doc.metadata, "store": "pdf", "score": float(score)}
        pdf_scored.append((doc, score))

    # Per-store quota: pin the top min_pdf PDF hits, then fill the rest by raw score.
    archive_scored.sort(key=lambda pair: pair[1])
    pdf_scored.sort(key=lambda pair: pair[1])

    quota = min(min_pdf, len(pdf_scored), k_total)
    pinned = pdf_scored[:quota]
    pool   = archive_scored + pdf_scored[quota:]
    pool.sort(key=lambda pair: pair[1])

    picks = pinned + pool[: k_total - quota]
    picks.sort(key=lambda pair: pair[1])
    return [doc for doc, _ in picks[:k_total]]


# Quick smoke test
hits = hybrid_retrieve("What is Amber?")
for i, d in enumerate(hits, 1):
    print(f"{i}. [{d.metadata.get('store')}] score={d.metadata.get('score'):.4f} "
          f"page={d.metadata.get('page', '-')}")
    print("   ", d.page_content[:160].replace("\n", " "), "...")

1. [archive] score=0.3206 page=-
    Steve Seibold < seibold.chemistry.msu.edu > (Wed, 5 May 2010 08:38:46 -0400): Hi Sorry if this is an ignorant question, but I have been reading on the Amber ema ...
2. [archive] score=0.3282 page=-
    via AMBER < amber.ambermd.org > (Mon, 12 Jun 2023 02:03:48 +0800 (CST)):  ...
3. [archive] score=0.3356 page=-
    < steinbrt.rci.rutgers.edu > (Thu, 28 Mar 2013 09:16:56 -0400 (EDT)): Hi,  > Thank you very much.. I am very new to amber. First time i am using&nbsp;  > follow ...
4. [archive] score=0.3400 page=-
    via AMBER < amber.ambermd.org > (Mon, 1 Jan 2024 00:25:24 +0800 (CST)):  ...
5. [pdf] score=0.7695 page=1
    Amber 2025 Reference Manual (Covers Amber24 and AmberTools25) ...


## 6) LLM + prompt

In [17]:
llm = ChatOllama(model=OLLAMA_MODEL, temperature=TEMPERATURE)

In [18]:
custom_prompt = ChatPromptTemplate.from_template("""You are AmberRAG, an expert AI assistant specializing in the AMBER molecular dynamics suite
and AmberTools workflows.

You answer questions strictly using the provided Context (archive discussions and manuals).

CORE RULES:
1) Use ONLY the provided Context to generate your answer.
2) Do NOT use outside knowledge or prior training information.
3) You may logically reason based on information in the Context,
   but do NOT introduce new facts that are not supported by it.
4) If the Context contains relevant information, use it to answer as completely as possible.
5) Do NOT mention any Persona, Identity, or Role in your answer.
6) Do NOT fabricate AMBER commands, flags, filenames, or parameter values.

STYLE:
- Start with a clear, direct answer.
- Then provide a concise technical explanation.
- Include practical AMBER-specific guidance only if supported by the Context.
- Address multiple sub-questions in the same order asked.
- Be precise, professional, and focused.
- Avoid unnecessary verbosity.

For source citations, include the source page number (for PDF) or source label (for archive) at the beginning.

Context:
{context}

Question: {question}

Answer:""")

## 7) Answer chain
Just `prompt → llm → parser`. Retrieval lives in `query_rag` so we don't pay for two trips to the vector stores per question.

In [19]:
def format_docs(docs: List[Document]) -> str:
    formatted = []
    for i, doc in enumerate(docs, 1):
        store = doc.metadata.get("store", "unknown")
        source = doc.metadata.get("source", "unknown_source")
        page = doc.metadata.get("page", "-")
        score = doc.metadata.get("score", None)
        score_str = f" | Score: {score:.4f}" if isinstance(score, (int, float)) else ""
        formatted.append(
            f"[Chunk {i} | Store: {store} | Source: {source} | Page: {page}{score_str}]\n"
            f"{doc.page_content}"
        )
    return "\n\n".join(formatted)

In [20]:
# A single answer chain that takes a pre-formatted context string and a question.
# We deliberately do NOT bake retrieval into the chain so query_rag can retrieve once
# and reuse the same docs for both the LLM call and the source display.
answer_chain = custom_prompt | llm | StrOutputParser()

## 8) Query helper — answer + the top-5 sources that fed it

In [21]:
def query_rag(question: str, k_total: int = K_TOTAL, save: bool = False):
    """Retrieve once, generate once, print answer + sources. Optionally append to RESULTS_PATH."""
    print(f"Question: {question}")
    print("-" * 60)

    docs = hybrid_retrieve(question, k_total=k_total)
    answer = answer_chain.invoke({
        "context": format_docs(docs),
        "question": question,
    })

    print("Answer:")
    print(answer)

    print("\nTop sources used:")
    for i, doc in enumerate(docs, 1):
        store = doc.metadata.get("store", "?")
        page = doc.metadata.get("page", "-")
        src = doc.metadata.get("source", "-")
        score = doc.metadata.get("score")
        score_str = f"{score:.4f}" if isinstance(score, (int, float)) else "-"
        print(f"\n--- Source {i} [{store}] page={page} src={src} score={score_str} ---")
        print(doc.page_content[:240].replace("\n", " "), "...")

    if save:
        record = {
            "question": question,
            "answer": answer,
            "sources": [
                {
                    "store": d.metadata.get("store"),
                    "page": d.metadata.get("page"),
                    "source": d.metadata.get("source"),
                    "score": d.metadata.get("score"),
                    "snippet": d.page_content[:500],
                }
                for d in docs
            ],
        }
        existing = []
        if os.path.exists(RESULTS_PATH):
            with open(RESULTS_PATH) as f:
                try:
                    existing = json.load(f)
                except json.JSONDecodeError:
                    existing = []
        existing.append(record)
        with open(RESULTS_PATH, "w") as f:
            json.dump(existing, f, indent=2)

    return answer, docs

## 9) Evaluation suite
All test questions in one list and one loop. Each Q/A pair (plus its top sources) is appended to `RESULTS_PATH` as JSON so you can diff against the ChatGPT and no-RAG notebooks structurally instead of by eyeballing scrollback.

In [22]:
EVAL_QUERIES = [
    "What is Amber?",
    "Why should SHAKE be disabled during minimization in AMBER?",
    "How do I obtain a Z-DNA structure from NAB?",
    "How can I get SHAKE to consider two different residue names to be water?",
    "Best OS recommendation for installing Amber + CUDA on an RTX 3070 server: "
    "Rocky Linux vs Ubuntu? What setup is known to work?",
    "How do I use paramfit to generate force field parameters for boron-containing compounds?",
    (
        "I have run a short TI simulation on a system using pmemd wherein, I have in "
        "one state bonded disulphide bridge (state A) and in the other unbonded bridge "
        "(State B). The peptide I am working with is 40 residues long. Can somebody "
        "kindly suggest me how do I extract pdb of the two states?"
    ),
    (
        "I'm using CPPTRAJ to analyze simulations of parallel strand DNA and want to "
        "get step parameters via nastruct. Does anyone know how nastruct identifies "
        "the molecule as parallel strands? If I write 'guessbp bptype para' in the "
        "nastruct command, the program will simply get stuck and look as if it is not "
        "continuing to run at all. If I just write 'guessbp', it would seem to treat "
        "the molecule as anti-parallel. What is the correct way to do that?"
    ),
]

# Wipe last run's results so we start clean.
if os.path.exists(RESULTS_PATH):
    os.remove(RESULTS_PATH)

for q in EVAL_QUERIES:
    query_rag(q, save=True)
    print("\n" + "=" * 72 + "\n")

print(f"All answers saved to {RESULTS_PATH}")

Question: What is Amber?
------------------------------------------------------------
Answer:
http://ambermd.org/pmwiki/ (Source: Carlos Simmerling, Wed, 5 May 2010 09:23:29 -0400)

The Amber Wiki refers to the developers' wiki, which can be accessed at http://ambermd.org/pmwiki/. This is a resource for important insights on compiling AMBER and AMBER toolset.

Technical Explanation:
The Amber Wiki is a web-based platform that provides documentation, tutorials, and resources for users of the AMBER molecular dynamics suite. It serves as a central hub for developers to share knowledge, best practices, and troubleshooting guides related to AMBER and its associated tools.

Practical Guidance:
To access the Amber Wiki, navigate to http://ambermd.org/pmwiki/. This will provide you with a comprehensive resource for learning about AMBER, including tutorials, documentation, and troubleshooting guides.

Top sources used:

--- Source 1 [archive] page=- src=- score=0.3206 ---
Steve Seibold < seibol